In [ ]:
# this notebook conduct the following task:
# 1 Extract Hchain Lchain and antigen chain from the csv info of # https://opig.stats.ox.ac.uk/webapps/sabdab-sabpred/sabdab/search/?all=true
#   structures_dir = "/content/drive/MyDrive/@antibody/antigen_epitope/data/structures/"
# 2 Extract antigen's structure info and interaction info, based on the antibody-antigen complex (two chains for antibody, one chain for antigen) from the previous step,
#   output_dir = "/content/drive/MyDrive/@antibody/antigen_epitope/data/antigen_info/"
# 3 TEST parts, use pymol to calculate the secondary structure and SaSaRate

In [ ]:
from google.colab import drive   # connect colab to google drive for file transferring
drive.mount('/content/drive')

In [ ]:
## for the first-time execution of this code
## install the packages in google Drive locally (/content/drive/MyDrive/Colab Notebooks/antigen_antibody/packages/)
import os
import sys

nb_path = '/content/notebooks'

if not os.path.exists(nb_path):
    os.symlink('/content/drive/MyDrive/@antibody/antigen_epitope/packages', nb_path)

sys.path.insert(0, nb_path)

In [ ]:
# first-time use execution
!pip install biopython --target=$nb_path

In [ ]:
!pip install scipy --target=$nb_path

In [ ]:
## non-first-time execution
## package was previously installed through process_data_positive_patch_3points-Graph.ipynb
import sys
sys.path.append('/content/drive/MyDrive/@antibody/antigen_epitope/packages')

In [ ]:
import os
import numpy as np
import pandas as pd
from Bio import PDB

In [ ]:
#@title Read csv file
# https://opig.stats.ox.ac.uk/webapps/sabdab-sabpred/sabdab/search/?all=true
# oneNote location: antigen_epitope


# Define the data directory
data_dir = '/content/drive/MyDrive/@antibody/antigen_epitope/data/'

# File name
file_name = 'sabdab_protein_summary.tsv'
file_path = data_dir + file_name

# Read the CSV file into a pandas DataFrame
df = pd.read_csv(file_path, delimiter='\t')

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

print(df.shape)
# Show the first few rows of the DataFrame
df.head(10)

In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [ ]:
# Remove rows where 'antigen_chain' is NaN
# Remove rows where 'antigen_chain' is NaN or empty
df = df.dropna(subset=['antigen_chain'])
df = df[df['antigen_chain'].str.strip() != '']
filtered_df = df.dropna(subset=['antigen_chain'])

print(filtered_df.shape)

In [ ]:
df= df.head(100)
df

In [ ]:
#@title  Extract Hchain Lchain and antigen chain from the csv info
# if Hchain and Lchain have the same id, only extrace once
# each complex has 3 or 2 chains, one is antibody, the other is antigen
# complexes are saved here:/content/drive/MyDrive/@antibody/antigen_epitope/data/structures/

from Bio import PDB
import pandas as pd
import os

# Initialize PDB parser and retriever
parser = PDB.PDBParser(QUIET=True)
pdbl = PDB.PDBList()

# Create a DataFrame from your data; replace this with your actual DataFrame
# df = ...

# Filter the DataFrame to include only necessary columns and drop rows with NA in 'antigen_chain'
filtered_df = filtered_df[['pdb', 'Hchain', 'Lchain', 'model', 'antigen_chain', 'antigen_type']].dropna(subset=['antigen_chain'])

# Create a temporary directory for downloaded PDB files
os.makedirs("/content/tmp", exist_ok=True)

# Iterate through each row in the DataFrame
for index, row in filtered_df.iterrows():
    try:
        pdb_id = row['pdb']
        H_chain = row['Hchain'].strip()
        L_chain = row.get('Lchain')
        if L_chain:
            L_chain = L_chain.strip()

        antigen_chains = [chain.strip() for chain in row['antigen_chain'].split('|')]
        antigen_types = [type_.strip() for type_ in row['antigen_type'].split('|')]

        selected_antigen_chains = [chain for chain, type_ in zip(antigen_chains, antigen_types) if type_ == 'protein']

        downloaded_file_path = pdbl.retrieve_pdb_file(pdb_id, pdir='/content/tmp', file_format='pdb', overwrite=True)
        if not downloaded_file_path:
            print(f"Desired structure {pdb_id} doesn't exist")
            continue

        print(f"Downloading PDB structure '{pdb_id}'...")

        structure = parser.get_structure(pdb_id, downloaded_file_path)

        for antigen_chain in selected_antigen_chains:
            print(f"Processing antigen chain {antigen_chain}")

            io = PDB.PDBIO()
            io.set_structure(structure)

            antibody_chains = [H_chain]
            if L_chain and L_chain != H_chain:
                antibody_chains.append(L_chain)

            class ChainSelector(PDB.Select):
                def accept_chain(self, chain):
                    is_antigen = chain.id.strip() == antigen_chain
                    is_antibody = chain.id.strip() in antibody_chains
                    is_accepted = is_antigen or is_antibody
                    print(f"Chain {chain.id} {'accepted' if is_accepted else 'rejected'}")
                    return is_accepted

            L_str = f"_{L_chain}" if L_chain else "_nan"
            output_path = f"/content/drive/MyDrive/@antibody/antigen_epitope/data/structures/{pdb_id}_{H_chain}{L_str}_{antigen_chain}.pdb"

            io.save(output_path, ChainSelector())

    except Exception as e:
        print(f"An error occurred while processing {pdb_id}: {e}")


In [ ]:
# in the directory /content/drive/MyDrive/@antibody/antigen_epitope/data/structures/

In [ ]:
# Not need to change some file names of the complex
# e.g.   7uoj_k_m_I(1).pdb into  7uoj_k_m_I.pdb
# in colab it shows: 7uoj_K_M_I.pdb 7uoj_k_m_I.pdb

In [ ]:
#@title Extract antigen's structure info and interaction info

from Bio import PDB
import pandas as pd
from pymol import cmd, stored
from scipy.spatial import distance
import os

# Function to calculate center of mass of sidechain
# Glycine (GLY) does not have a side chain beyond its alpha carbon, so its side chain center of mass would be undefined in your code
def calculate_center_of_mass(residue):
    if residue.resname == "GLY":
        return residue['CA'].coord  # Return alpha carbon coordinates for Glycine

    mass = 0
    x, y, z = 0, 0, 0
    backbone_atoms = ['N', 'CA', 'C', 'O']
    for atom in residue.get_unpacked_list():
        if atom.name not in backbone_atoms:
            mass += atom.mass
            x += atom.coord[0] * atom.mass
            y += atom.coord[1] * atom.mass
            z += atom.coord[2] * atom.mass
    if mass == 0:
        return 0, 0, 0
    return x / mass, y / mass, z / mass


# Process each PDB file in the directory
structures_dir = "/content/drive/MyDrive/@antibody/antigen_epitope/data/structures/"
output_dir = "/content/drive/MyDrive/@antibody/antigen_epitope/data/antigen_info/"

for pdb_file in os.listdir(structures_dir):
    if pdb_file.endswith('.pdb'):
        pdb_file_path = os.path.join(structures_dir, pdb_file)

        # Extract chain information
        chains = pdb_file.split('_')[1:-1]  # Extract the chain names (e.g., ['H', 'L'] or ['S'])
        antigen_chain = pdb_file.split('_')[-1][0]  # Extract antigen chain (e.g., 'X')

        # Load the PDB file into PyMOL
        cmd.reinitialize()  # Clear previous data
        cmd.load(pdb_file_path, "protein_complex")

        # Calculate relative SASA
        SaSaRate = cmd.get_sasa_relative("protein_complex and chain " + antigen_chain)
        sasa_values = [value for key, value in SaSaRate.items()]

        # Initialize an empty list for secondary structure
        stored.ss_values = []
        cmd.iterate("protein_complex and chain " + antigen_chain + " and name CA", "stored.ss_values.append(ss)")

        # Initialize PDB Parser
        parser = PDB.PDBParser(QUIET=True)

        # Load structure
        structure = parser.get_structure(pdb_file.split('.')[0], pdb_file_path)

        # Initialize DataFrame
        columns = ['name', 'ResID', 'AA', 'SS', 'Cax', 'Cay', 'Caz', 'SCx', 'SCy', 'SCz', 'SaSaRate',
                   'G', 'A', 'V', 'I', 'L', 'S', 'T', 'D', 'N', 'E', 'Q', 'K', 'R', 'C', 'M', 'F', 'Y', 'W', 'H', 'P']
        df = pd.DataFrame(columns=columns)

        # Prepare a list for antibody sidechain centers
        antibody_centers = []

        # Populate antibody_centers
        for model in structure:
            for chain in model:
                if chain.id in chains:
                    for residue in chain:
                        if "CA" in residue:
                            sc_x, sc_y, sc_z = calculate_center_of_mass(residue)
                            ab_resname = residue.resname
                            antibody_centers.append((ab_resname, (sc_x, sc_y, sc_z)))

        # Iterate through structure to get antigen residues
        for model in structure:
            for chain in model:
                if chain.id == antigen_chain:
                    for i, residue in enumerate(chain):
                        if "CA" in residue:
                            c_alpha = residue['CA'].coord
                            sc_x, sc_y, sc_z = calculate_center_of_mass(residue)
                            #ss = stored.ss_values[i]
                            # Safeguard against IndexError
                            ss = stored.ss_values[i] if i < len(stored.ss_values) else 'Unknown'

                            interaction_data = {aa: 0 for aa in ['G', 'A', 'V', 'I', 'L', 'S', 'T', 'D', 'N', 'E', 'Q', 'K', 'R', 'C', 'M', 'F', 'Y', 'W', 'H', 'P']}
                            antigen_center = (sc_x, sc_y, sc_z)

                            for ab_resname, ab_center in antibody_centers:
                                if distance.euclidean(antigen_center, ab_center) <= 7:
                                  try:
                                    one_letter_code = PDB.Polypeptide.protein_letters_3to1[ab_resname]
                                    interaction_data[one_letter_code] = 1
                                  except KeyError:
                                    print(f"Warning: Unexpected residue name '{ab_resname}' encountered.")

                            data = {'name': pdb_file.split('.')[0],
                                    'ResID': residue.id[1],  # Residue number
                                    'AA': residue.resname,
                                    'SS': ss,
                                    'Cax': c_alpha[0],
                                    'Cay': c_alpha[1],
                                    'Caz': c_alpha[2],
                                    'SCx': sc_x,
                                    'SCy': sc_y,
                                    'SCz': sc_z,
                                    'SaSaRate': sasa_values[i]}
                            data.update(interaction_data)
                            df = pd.concat([df, pd.DataFrame([data])], ignore_index=True)

        # Save DataFrame to TSV
        output_path = os.path.join(output_dir, f"{pdb_file.split('.')[0]}_info.tsv")
        df.to_csv(output_path, sep='\t', index=False)

        print(f"Processed {pdb_file} and saved to {output_path}")



In [ ]:
# how many tsv files processed
import os

output_dir = "/content/drive/MyDrive/@antibody/antigen_epitope/data/antigen_info/"

# List the files in the directory
files_in_directory = os.listdir(output_dir)

# Count the number of .tsv files
number_of_tsv_files = len([f for f in files_in_directory if f.endswith('.tsv')])

print(f"There are {number_of_tsv_files} .tsv files in {output_dir}.")


In [ ]:
#@title TEST - generate the antigen info data frame
# this antigen data frame contains the structure info and interaction info

from Bio import PDB
import pandas as pd
from pymol import cmd, stored
from scipy.spatial import distance

# Function to calculate center of mass of sidechain
def calculate_center_of_mass(residue):
    mass = 0
    x, y, z = 0, 0, 0
    backbone_atoms = ['N', 'CA', 'C', 'O']
    for atom in residue.get_unpacked_list():
        if atom.name not in backbone_atoms:
            mass += atom.mass
            x += atom.coord[0] * atom.mass
            y += atom.coord[1] * atom.mass
            z += atom.coord[2] * atom.mass
    if mass == 0:
        return 0, 0, 0
    return x / mass, y / mass, z / mass

# Load the PDB file into PyMOL
pdb_file_path = "/content/drive/MyDrive/@antibody/antigen_epitope/data/structures/1ors_B_A_C.pdb"
cmd.load(pdb_file_path, "protein_complex")

# Calculate relative SASA
SaSaRate = cmd.get_sasa_relative("protein_complex and chain C")
sasa_values = [value for key, value in SaSaRate.items()]

# Initialize an empty list for secondary structure
stored.ss_values = []
cmd.iterate("protein_complex and chain C and name CA", "stored.ss_values.append(ss)")

# Initialize PDB Parser
parser = PDB.PDBParser(QUIET=True)

# Load structure
structure = parser.get_structure('1ors_B_A_C', pdb_file_path)

# Initialize DataFrame
columns = ['name', 'ResID', 'AA', 'SS', 'Cax', 'Cay', 'Caz', 'SCx', 'SCy', 'SCz', 'SaSaRate',
           'G', 'A', 'V', 'I', 'L', 'S', 'T', 'D', 'N', 'E', 'Q', 'K', 'R', 'C', 'M', 'F', 'Y', 'W', 'H', 'P']
df = pd.DataFrame(columns=columns)

# Prepare a list for antibody sidechain centers
antibody_centers = []

# Populate antibody_centers
for model in structure:
    for chain in model:
        if chain.id in ["H", "L"]:
            for residue in chain:
                if "CA" in residue:
                    sc_x, sc_y, sc_z = calculate_center_of_mass(residue)
                    ab_resname = residue.resname
                    antibody_centers.append((ab_resname, (sc_x, sc_y, sc_z)))

# Iterate through structure to get antigen residues
for model in structure:
    for chain in model:
        if chain.id == "X":
            for i, residue in enumerate(chain):
                if "CA" in residue:
                    c_alpha = residue['CA'].coord
                    sc_x, sc_y, sc_z = calculate_center_of_mass(residue)
                    ss = stored.ss_values[i]

                    interaction_data = {aa: 0 for aa in ['G', 'A', 'V', 'I', 'L', 'S', 'T', 'D', 'N', 'E', 'Q', 'K', 'R', 'C', 'M', 'F', 'Y', 'W', 'H', 'P']}
                    antigen_center = (sc_x, sc_y, sc_z)

                    for ab_resname, ab_center in antibody_centers:
                        if distance.euclidean(antigen_center, ab_center) <= 7:
                            #one_letter_code = PDB.Polypeptide.three_to_one(ab_resname)
                            one_letter_code = PDB.Polypeptide.protein_letters_3to1[ab_resname]
                            interaction_data[one_letter_code] = 1

                    data = {'name': '1ors_B_A_C',
                            'ResID': residue.id[1],  # Residue number
                            'AA': residue.resname,
                            'SS': ss,
                            'Cax': c_alpha[0],
                            'Cay': c_alpha[1],
                            'Caz': c_alpha[2],
                            'SCx': sc_x,
                            'SCy': sc_y,
                            'SCz': sc_z,
                            'SaSaRate': sasa_values[i]}
                    data.update(interaction_data)
                    df = pd.concat([df, pd.DataFrame([data])], ignore_index=True)

# Save DataFrame to TSV
output_path = "/content/drive/MyDrive/@antibody/antigen_epitope/data/antigen_info/1ors_B_A_C_info.tsv"
df.to_csv(output_path, sep='\t', index=False)

In [ ]:
df.tail(10)

In [ ]:
!apt-get install -y python3-pymol

In [ ]:
#@title TEST -- extract antigen chain and calculate its SaSaRate
# Import the PyMOL command module
from pymol import cmd

# Load the PDB file into PyMOL
pdb_file_path = "/content/drive/MyDrive/@antibody/antigen_epitope/data/structures/7xrz_H_L_X.pdb"
cmd.load(pdb_file_path, "protein_complex")

# Select chain C
cmd.select("chainX", "protein_complex and chain X")

# Calculate the relative SASA
SaSaRate = cmd.get_sasa_relative("chainX")
print(SaSaRate)

sasa_values = []
# Iterate over the defaultdict to get just the float values
for key, value in SaSaRate.items():
    sasa_values.append(value)
print(sasa_values)

# Iterate over the defaultdict to get just the float values
for key, sasa_value in SaSaRate.items():
    print(key, sasa_value)

size_of_dictionary = len(SaSaRate)
print(f"The size of the SaSaRate dictionary is {size_of_dictionary}")

In [ ]:
#@title TEST -- secondary structure
from pymol import cmd, stored

# Initialize an empty list to store secondary structure data in the stored namespace
stored.ss_values = []

# Load the PDB file into PyMOL
pdb_file_path = "/content/drive/MyDrive/@antibody/antigen_epitope/data/structures/7xrz_H_L_X.pdb"
cmd.load(pdb_file_path, "protein_complex")

# Select chain X
cmd.select("chainX", "protein_complex and chain X")

# Use cmd.iterate to access the 'ss' (secondary structure) attribute of each residue
cmd.iterate("chainX and name CA", "stored.ss_values.append((resi, ss))")

# Print the secondary structure for each residue
#for resi, ss in stored.ss_values:
#    print(f"Residue {resi}: Secondary Structure = {ss}")

stored.ss_values = []
cmd.iterate("protein_complex and chain X and name CA", "stored.ss_values.append(ss)")
print(stored.ss_values)

In [ ]:
#@title TEST -- extract the antigen chain C and add SaSaRate to the b-factor of each residue

from Bio.PDB import PDBParser, PDBIO

# Initialize the PDB parser and read the file
parser = PDBParser(QUIET=True)
structure = parser.get_structure('protein_complex', '/content/drive/MyDrive/@antibody/antigen_epitope/data/structures/8goc_M_N_C.pdb')

# Create an empty Structure to save the chain C
from Bio.PDB.Structure import Structure
chain_C_structure = Structure('chain_C_structure')

# Add a model to the new structure
from Bio.PDB.Model import Model
model = Model(0)
chain_C_structure.add(model)

# Get chain C from the original structure and add it to the new structure
chain_C = structure[0]['C']
model.add(chain_C)

# Update B-factors based on the SaSaRate (you should already have this from your PyMOL calculations)
# Here, we're assuming SaSaRate is a dictionary with keys being the residue numbers and values being the calculated SASA
# SaSaRate = {...}

for residue in chain_C:
    resi = residue.get_id()[1]
    sasa_value = SaSaRate.get(('protein_complex', '', 'C', str(resi)), None)
    if sasa_value is not None:
        for atom in residue:
            atom.set_bfactor(sasa_value)

# Save the new structure as a PDB file
io = PDBIO()
io.set_structure(chain_C_structure)
io.save('/content/drive/MyDrive/@antibody/antigen_epitope/data/structures_antigen/8goc_C.pdb')
